# DE pseudobulk

Run donor-level pseudobulk differential expression for broad classes, selected glia subtypes, and selected excitatory subtypes. The contrast label follows the saved file name, while the log2 fold-change is interpreted as the second condition relative to the first condition.


In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


In [8]:
PROJECT_ROOT = Path("/nemo/lab/schreibera/home/users/wangj/FlowCytometry/c9_multiomics")
PSEUDOBULK_DIR = PROJECT_ROOT / "results" / "10_Pseudobulk_26donors"
DE_DIR = PROJECT_ROOT / "results" / "13_DE_pseudobulk_26donors"

INPUT_DIRS = {
    'broad': PSEUDOBULK_DIR / 'broad',
    'glia_subtypes': PSEUDOBULK_DIR / 'glia_subtypes',
    'excitatory_subtypes': PSEUDOBULK_DIR / 'excitatory_subtypes',
}
OUTPUT_DIRS = {
    'broad': DE_DIR / 'broad',
    'glia_subtypes': DE_DIR / 'glia_subtypes',
    'excitatory_subtypes': DE_DIR / 'excitatory_subtypes',
}
for out_dir in OUTPUT_DIRS.values():
    out_dir.mkdir(parents=True, exist_ok=True)


## Settings


In [15]:
RUN_DE = True
OVERWRITE_EXISTING_RESULTS = False
MIN_DONORS_PER_CONDITION = 2
MIN_CELLS_PER_DONOR_GROUP = 5

GROUPS = {
    'broad': ['Excitatory', 'Inhibitory', 'Glia', 'Vascular'],
    'glia_subtypes': ['Glia_Oligo', 'Glia_OPC', 'Glia_Micro', 'Glia_Astro'],
    'excitatory_subtypes': [
        'Ex_L5_PCP4_NXPH2',
        'Ex_L5_PCP4_NXPH2_UMN_like',
        'Ex_L5_L6_THEMIS_NR4A2',
        'Ex_L5_L6_THEMIS_TMEM233',
        'Ex_L6_TLE4_CCBE1',
        'Ex_L6_TLE4_MEGF11',
    ],
}

# Files are labelled first_vs_second; PyDESeq2 contrast is second relative to first.
COMPARISONS = [
    ('Control', 'sALS'),
    ('Control', 'c9ALS'),
    ('sALS', 'c9ALS'),
]

In [16]:
umn_counts_path = INPUT_DIRS['excitatory_subtypes'] / 'Ex_L5_PCP4_NXPH2_UMN_like_counts.csv'
umn_meta_path = INPUT_DIRS['excitatory_subtypes'] / 'Ex_L5_PCP4_NXPH2_UMN_like_meta.csv'

print('UMN-like pseudobulk counts exists:', umn_counts_path.exists())
print('UMN-like pseudobulk meta exists:', umn_meta_path.exists())

if umn_meta_path.exists():
    umn_meta = pd.read_csv(umn_meta_path, index_col=0)
    display(
        umn_meta.groupby('condition')
        .agg(
            n_donors=('condition', 'size'),
            min_cells=('n_cells', 'min'),
            median_cells=('n_cells', 'median'),
            max_cells=('n_cells', 'max'),
        )
    )

UMN-like pseudobulk counts exists: True
UMN-like pseudobulk meta exists: True


,n_donors,min_cells,median_cells,max_cells
condition,,,,
Control,2,6,7.5,9
c9ALS,2,6,10.5,15


## Load pseudobulk and inspect donor coverage


In [17]:
def load_pseudobulk(group_name, input_dir):
    counts = pd.read_csv(input_dir / f'{group_name}_counts.csv', index_col=0)
    meta = pd.read_csv(input_dir / f'{group_name}_meta.csv', index_col=0)
    counts.index = counts.index.astype(str)
    meta.index = meta.index.astype(str)
    common = counts.index.intersection(meta.index)
    counts = counts.loc[common]
    meta = meta.loc[common].copy()
    return counts, meta

coverage_rows = []
available_groups = {level: [] for level in GROUPS}

for level, group_names in GROUPS.items():
    for group_name in group_names:
        counts_path = INPUT_DIRS[level] / f'{group_name}_counts.csv'
        meta_path = INPUT_DIRS[level] / f'{group_name}_meta.csv'
        if not counts_path.exists() or not meta_path.exists():
            coverage_rows.append({'level': level, 'group': group_name, 'condition': pd.NA, 'n_donors': 0, 'available': False})
            continue

        counts, meta = load_pseudobulk(group_name, INPUT_DIRS[level])
        if 'n_cells' in meta.columns:
            meta = meta[meta['n_cells'] >= MIN_CELLS_PER_DONOR_GROUP]
            counts = counts.loc[meta.index]
        available_groups[level].append(group_name)
        for condition, n_donors in meta['condition'].astype(str).value_counts().sort_index().items():
            coverage_rows.append({'level': level, 'group': group_name, 'condition': condition, 'n_donors': int(n_donors), 'available': True})

coverage = pd.DataFrame(coverage_rows)
display(coverage.pivot_table(index=['level', 'group'], columns='condition', values='n_donors', fill_value=0, aggfunc='sum'))


condition                                      Control  c9ALS  sALS
level               group                                          
broad               Excitatory                      10      6    10
                    Glia                            10      6    10
                    Inhibitory                       9      6    10
                    Vascular                        10      6    10
excitatory_subtypes Ex_L5_L6_THEMIS_NR4A2            8      6     8
                    Ex_L5_L6_THEMIS_TMEM233          9      6    10
                    Ex_L5_PCP4_NXPH2                 7      6     8
                    Ex_L5_PCP4_NXPH2_UMN_like        2      2     0
                    Ex_L6_TLE4_CCBE1                 9      6     9
                    Ex_L6_TLE4_MEGF11                9      6     9
glia_subtypes       Glia_Astro                      10      6    10
                    Glia_Micro                      10      6    10
                    Glia_OPC                        10      6    10
                    Glia_Oligo                      10      6    10

## Run DESeq2 contrasts


In [18]:
def run_deseq2(counts, meta, numerator, denominator):
    keep = meta['condition'].astype(str).isin([denominator, numerator])
    counts_use = counts.loc[keep].round().astype(int)
    meta_use = meta.loc[keep, ['condition']].copy()
    meta_use['condition'] = pd.Categorical(meta_use['condition'].astype(str), categories=[denominator, numerator])

    donor_counts = meta_use['condition'].value_counts()
    if (donor_counts < MIN_DONORS_PER_CONDITION).any():
        return None, f'too_few_donors: {donor_counts.to_dict()}'

    expressed = counts_use.sum(axis=0) > 0
    counts_use = counts_use.loc[:, expressed]
    if counts_use.shape[1] == 0:
        return None, 'no_expressed_genes'

    dds = DeseqDataSet(counts=counts_use, metadata=meta_use, design='~condition', refit_cooks=True)
    dds.deseq2()
    stats = DeseqStats(dds, contrast=['condition', numerator, denominator])
    stats.summary()
    result = stats.results_df.reset_index()
    result = result.rename(columns={result.columns[0]: 'gene'})
    return result, None

skipped = []
written = []

if RUN_DE:
    for level, group_names in available_groups.items():
        for group_name in group_names:
            counts, meta = load_pseudobulk(group_name, INPUT_DIRS[level])
            if 'n_cells' in meta.columns:
                meta = meta[meta['n_cells'] >= MIN_CELLS_PER_DONOR_GROUP]
                counts = counts.loc[meta.index]

            for denominator, numerator in COMPARISONS:
                comparison = f'{denominator}_vs_{numerator}'
                out_path = OUTPUT_DIRS[level] / f'{group_name}_{comparison}.csv'
                if out_path.exists() and not OVERWRITE_EXISTING_RESULTS:
                    written.append({'level': level, 'group': group_name, 'comparison': comparison, 'path': str(out_path), 'status': 'already_exists'})
                    continue

                result, reason = run_deseq2(counts, meta, numerator=numerator, denominator=denominator)
                if reason:
                    skipped.append({'level': level, 'group': group_name, 'comparison': comparison, 'reason': reason})
                    continue

                result.to_csv(out_path, index=False)
                written.append({'level': level, 'group': group_name, 'comparison': comparison, 'path': str(out_path), 'status': 'written'})

written = pd.DataFrame(written)
skipped = pd.DataFrame(skipped)
display(written)
if not skipped.empty:
    display(skipped)


Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 3.72 seconds.

Fitting dispersion trend curve...
/nemo/lab/schreibera/home/users/wangj/.conda/envs/c9_multiomics_py311/lib/python3.11/site-packages/pydeseq2/dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.33 seconds.

/nemo/lab/schreibera/home/users/wangj/.conda/envs/c9_multiomics_py311/lib/python3.11/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 5.09 seconds.

Fitting LFCs...
... done in 5.45 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 3.81 seconds.



Log2 fold change & Wald test p-value: condition c9ALS vs Control
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
ATAD3B            2.098893        0.062658  1.771932  0.035362  0.971791   
ENSG00000234396   1.322770       -3.774635  3.689867 -1.022973  0.306321   
PRDM16            0.132957       -1.245312  3.577779 -0.348069  0.727789   
MTND2P28          0.146367        1.321181  3.562259  0.370883  0.710725   
ACAP3             4.783398       -0.227885  1.632960 -0.139553  0.889013   
...                    ...             ...       ...       ...       ...   
ENSG00000278673   0.573881       -2.489267  3.172694 -0.784591  0.432693   
ENSG00000276256   0.416703        1.827011  3.637111  0.502325  0.615439   
ENSG00000273748  12.487852       -0.786392  1.562746 -0.503212  0.614815   
ENSG00000278817   0.545239       -1.121169  2.615745 -0.428623  0.668197   
ENSG00000271254   0.563070        2.526422  3.144310  0.803490  0.421692   

                     p

,level,group,comparison,path,status
0,broad,Excitatory,Control_vs_sALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
1,broad,Excitatory,Control_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
2,broad,Excitatory,sALS_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
3,broad,Inhibitory,Control_vs_sALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
4,broad,Inhibitory,Control_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
5,broad,Inhibitory,sALS_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
6,broad,Glia,Control_vs_sALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
7,broad,Glia,Control_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
8,broad,Glia,sALS_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists
9,broad,Vascular,Control_vs_sALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...,already_exists


,level,group,comparison,reason
0,excitatory_subtypes,Ex_L5_PCP4_NXPH2_UMN_like,Control_vs_sALS,"too_few_donors: {'Control': 2, 'sALS': 0}"
1,excitatory_subtypes,Ex_L5_PCP4_NXPH2_UMN_like,sALS_vs_c9ALS,"too_few_donors: {'c9ALS': 2, 'sALS': 0}"


## Output inventory


In [19]:
inventory = []
for level, out_dir in OUTPUT_DIRS.items():
    for path in sorted(out_dir.glob('*.csv')):
        parts = path.stem.rsplit('_', 3)
        comparison = '_'.join(parts[-3:]) if len(parts) >= 4 else pd.NA
        group_name = path.stem[:-(len(comparison) + 1)] if pd.notna(comparison) else path.stem
        inventory.append({'level': level, 'group': group_name, 'comparison': comparison, 'path': str(path)})

inventory = pd.DataFrame(inventory)
inventory.to_csv(DE_DIR / 'de_output_inventory.csv', index=False)
display(inventory)


,level,group,comparison,path
0,broad,Excitatory,Control_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
1,broad,Excitatory,Control_vs_sALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
2,broad,Excitatory,sALS_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
3,broad,Glia,Control_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
4,broad,Glia,Control_vs_sALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
5,broad,Glia,sALS_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
6,broad,Inhibitory,Control_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
7,broad,Inhibitory,Control_vs_sALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
8,broad,Inhibitory,sALS_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...
9,broad,Vascular,Control_vs_c9ALS,/nemo/lab/schreibera/home/users/wangj/FlowCyto...


## Interpretation note

For a file named `Control_vs_c9ALS`, the contrast is `c9ALS` relative to `Control`. A positive log2FoldChange means higher expression in c9ALS than Control; a negative value means lower expression in c9ALS than Control.
